In [ ]:
from libraries.inference_training import Configuration, ImageDataset
from libraries.inference_training import initCudaEnvironment, createTransforms
from libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from libraries.inference_training import trainModel, saveModel, loadModel
import random

In [ ]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

In [ ]:
def createModel(trainDirectory: str, testDirectory: str, modelName: str, epochs: int, labels: list[str], augment_data: bool, save_path: str, save_interval=0, maskdata=None , model_description="default"):
    """
    :param trainDirectory: path naar training dataset
    :param testDirectory: path naar testing dataset
    :param modelName: naam van model
    :param epochs: hoeveelheid epochs
    :param labels: list van labels, geef normaal ["parkeerplaatsen"] als er geen andere objecten zijn
    :param augment_data: bepaald of er image transforms gedaan worden, nog niet getest
    :param save_path: path naar save locatie van model
    :param maskdata: list van floats 
    :param save_interval: 
    :param model_description: beschrijft het model in de ONNX als het gesaved is
    :return:
    """
    if maskdata is None:
        maskdata = [0.2, 0.3, 0.5]
    config = Configuration()
    print("Device: " + str(config.device))
    config.setSaveInterval(save_interval)
    config.setSavePath(save_path)
    config.setIsCrowd(False)
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName(modelName)
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2 + 1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(epochs)
    config.setOnnxInfo(producer="Tygron", description=model_description)
    config.addLegendEntry("Background", 0, "#00000000")
    i = 1
    for label in labels:
        config.addLegendEntry(label, i, "#" + ''.join([random.choice('ABCDEF0123456789') for i in range(6)]))
        i += 1

    config.setOnnxMetaData(scoreThreshold=maskdata[0],
                           maskThreshold=maskdata[1],
                           strideFraction=maskdata[2])

    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    if augment_data:
        trainingDataset = ImageDataset(config, True, imageTransforms=createTransforms(True))
        testDataset = ImageDataset(config, False, createTransforms(False))
    else:
        trainingDataset = ImageDataset(config, True, createTransforms(False))
        testDataset = ImageDataset(config, False, createTransforms(False))

    print("Train Image count: " + str(trainingDataset.__len__()))
    print("Test Image count: " + str(testDataset.__len__()))

    if not trainingDataset.validateFiles():
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles()

    if not testDataset.validateFiles():
        print("Inconsistent test dataset ")
        testDataset.validateFiles()

    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())

    model = trainModel(config, trainingDataset, testDataset)
    model.eval()

    saveModel(config, model, epoch=epochs)

    exportOnnxModel(config, model)
    writeONNXMeta(config)

    return model, config

# create model template

In [ ]:
train_directory = "<insert train path here>"
test_directory = "<insert test path here>"
modelName = "<insert name here>"
epochs = 1
labels = ["<insert labels here>"]
augment = False
save_path = "<insert path to save location for model here>"
save_Interval = 0 # model is saved in between these amount of epochs
createModel(train_directory, test_directory, modelName, epochs, labels, augment, save_path, save_Interval)

# combo model with augment

In [ ]:
train_directory = "C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/datasets/combo_overlay_sets/train"
test_directory = "C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/datasets/combo_overlay_sets/test"
modelName = "combo_model_with_new_augment"
epochs = 15
labels = ["parking_space"]  
augment = True
save_path = "C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/models/combo_models/augment/"
save_Interval = 5
model, config = createModel(train_directory, test_directory, modelName, epochs, labels, augment, save_path, save_Interval)

^ AP and AR converge to the results at the 15th epoch and don't change after with the current dataset.